# Torch replay of sampled V6-style policies on resident execution tensors

Third notebook: no market encoder/features, no teacher labels, no real V6 network.
A dummy output provider supplies masked actor heads and a constant-zero critic.
Exactly **one inference and one sampled proposal per simulated second** drives a
causal tensor environment. Existing actions execute over the next 1s liquidity
interval, with partial fills and remaining quantities. Data/state stay on the
selected device; CPU is an explicit fallback when CUDA is unavailable.

Compatibility here means the V6 **action/likelihood/observation contract**, not
drop-in OMS or profitability equivalence. We retain stable listing identities,
compact held slots, sampled latent evidence and actual equity-change rewards.

The current `research/rl_trading/v6/rollout.py::collect_session` can infer up to
64 times at a 1s clock until HOLD. This notebook follows the user's NEW one-call
cadence. Switching V6 to that cadence is a deliberate policy-collection change.
V6's existing quote-bound entry is first-following-100ms IOC, not this persistent
1s volume/VWAP model. Neither existing collector nor environment is modified.


## Agent integration map: verified current V6 code

* `v6/actor_critic.py::HybridDistribution`: masked logits `[1+N+3H]`, conditional
  Gaussian locations/scales; ENTER uses sigmoid(latent), STOP/TARGET softplus(latent).
  Store **token + latent + joint old_log_prob + old_value**, not just executed size.
* Axis: HOLD=0; ENTER=1..N; EXIT=`1+N+slot`; STOP=`1+N+H+slot`;
  TARGET=`1+N+2H+slot`. Slots index the **pre-decision** compact `held_index [H]`.
  H changes after fills. Refreshing a holding axis must never rekey listing identity.
* `v6/account_observation.py`: account `[7]` is cash, equity, realized, exposure/equity,
  seconds since last action, reserved cash, pending-entry count. Held `[H,11]` is
  shares, entry price, age, return, stop/mark gap, target/mark gap, has-stop,
  has-target, stop-pending, modeled-pause flag, modeled-pause age. Pause fields are
  explicitly zero here; a real adapter must restore LULD/halts and valid-mark rules.
* `v6/rollout.py`: remembers **actual execution outcomes** in the GRU. A GPU
  integration must feed outcomes back before the next inference, and separately
  preserve immediate bracket-setting outcomes. The dummy provider has no GRU.
* `v6/rl_training.py::PolicyRollout`: detached rewards, old likelihoods/values,
  elapsed_seconds, terminated, bootstrap. `update_on_policy` reconstructs current
  likelihoods chronologically; old hidden states cannot replace burn-in/rebuild.
* `v6/actor_critic.py::elapsed_gae`: discount by actual elapsed time. Our sampled
  steps are exactly 1s apart. If the 64-action cadence is retained, same-clock
  samples instead have zero elapsed time and inference count is not seconds.

No strategy rules run in place of the actor. Environment-only feasibility masks,
bracket execution and allocation still exist: a model proposal is not a fill.


In [ ]:
import os, sys, math
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import torch
from torch import nn
from torch.nn import functional as F

# Notebook imports must never create source-tree __pycache__ files.
os.environ["PYTHONDONTWRITEBYTECODE"]="1"
sys.dont_write_bytecode=True
REPO_ROOT=Path(r"D:/TradingCodes/quant-research-workbench")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0,str(REPO_ROOT))

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
MONEY_DTYPE=torch.float64  # Match accounting first; changing to float32 needs separate validation.
CPU_THREADS=1
if DEVICE.type=="cpu":
    torch.set_num_threads(CPU_THREADS)
print({"torch":torch.__version__,"device":str(DEVICE),"cpu_threads":torch.get_num_threads(),
       "cuda_measured_here":DEVICE.type=="cuda"})
PARTICIPATION=0.10
PRICE_INCREMENT=0.0001  # Explicit research precision, not an exchange tick-size claim.

def zeros_state(n,cash,device):
    # Listing-indexed state [N]; integers stay separate from money tensors.
    z=torch.zeros(n,device=device,dtype=MONEY_DTYPE)
    q=torch.zeros(n,device=device,dtype=torch.long)
    return dict(quantity=q,book_cost=z,stop=z.clone(),target=z.clone(),
                side=q.clone(),remaining=q.clone(),entry_cap=z.clone(),request_fraction=z.clone(),
                entry_second=torch.full_like(q,-1),last_action_second=q.new_zeros(()),
                cash=z.new_tensor(cash),realized=z.new_zeros(()))

def make_tape(n,seconds,device):
    # Execution-only tensors [T+1,N]. Row 0 is the decision's initial mark.
    # Row t+1 describes eligible volume/VWAP over (t,t+1], known only at its close.
    t=torch.arange(seconds+1,device=device,dtype=torch.long)[:,None]
    listing=torch.arange(n,device=device,dtype=torch.long)[None,:]
    mark=(20.0+(listing%100).to(MONEY_DTYPE)*0.1
          +((t+listing%5)%12-6).to(MONEY_DTYPE)*0.04)
    volume=torch.where((t+listing)%7==0,0.0,20.0).to(MONEY_DTYPE)
    return dict(mark=mark,vwap=mark.clone(),volume=volume,
                clocks=torch.arange(seconds+1,device=device,dtype=torch.long))

def observe(state,mark,second):
    qty=state["quantity"]
    # Exact V6 compact H axis. nonzero has data-dependent size and can synchronize
    # CUDA; benchmark includes this cost. A padded axis would need an explicit
    # new action encoding/adapter, not a silent substitution for 1+N+3H.
    held=torch.nonzero(qty>0,as_tuple=False).squeeze(1)
    average=state["book_cost"]/qty.clamp_min(1)
    equity=state["cash"]+(qty*mark).sum()
    reserved=torch.where(state["side"]==1,state["remaining"]*state["entry_cap"],0.0).sum()
    account=torch.stack((state["cash"],equity,state["realized"],
        (qty*mark).sum()/equity.clamp_min(1e-9),
        (second-state["last_action_second"]).to(MONEY_DTYPE),reserved,
        ((state["side"]==1)&(state["remaining"]>0)).sum().to(MONEY_DTYPE)))
    features=torch.stack((qty.to(MONEY_DTYPE),average,
        (second-state["entry_second"]).to(MONEY_DTYPE),
        (mark-average)/average.clamp_min(1e-9),
        torch.where(state["stop"]>0,(mark-state["stop"])/mark,0.0),
        torch.where(state["target"]>0,(state["target"]-mark)/mark,0.0),
        (state["stop"]>0).to(MONEY_DTYPE),(state["target"]>0).to(MONEY_DTYPE),
        (state["side"]==2).to(MONEY_DTYPE),torch.zeros_like(mark),torch.zeros_like(mark)),dim=1)[held]
    masks=dict(enter_allowed=(qty==0)&(state["remaining"]==0)&(account[0]-account[5]>0),
        exit_allowed=state["side"][held]!=2,
        stop_allowed=(state["stop"][held]==0)&(state["side"][held]!=2),
        target_allowed=(state["target"][held]==0)&(state["side"][held]!=2))
    return held,account.to(torch.float32),features.to(torch.float32),masks


## Dummy inference and V6 hybrid sampling

One provider forward occurs at every decision second, AFTER observation and
BEFORE sampling. It supplies V6-shaped heads, not market features. Family-balanced
logits keep ENTER from dominating solely because N is large. This is not a timing
proxy for the real encoder, ranking, attention, GRU and critic. The zero critic
also has exact zero bootstrap at a truncated final state; real V6 needs a causal
next-state value evaluation and its extra bootstrap-inference cost recorded.

The Gaussian latent and Jacobian are retained even when an entry fills zero
shares. Environment rounding/partial fills must never change PPO sample evidence.


In [ ]:
class DummyV6Outputs(nn.Module):
    def forward(self,n,held,account,held_features,masks):
        h=held.numel()
        # Scalar heads use float32 like the actor; accounting uses float64.
        z=account.new_zeros(1)
        logits=torch.cat((z,account.new_full((n,),0.4-math.log(n)),
            account.new_full((h,),0.2-math.log(max(h,1))),
            account.new_full((h,),0.3-math.log(max(h,1))),
            account.new_full((h,),0.3-math.log(max(h,1)))))
        allowed=torch.cat((torch.ones(1,device=account.device,dtype=torch.bool),
            masks["enter_allowed"],masks["exit_allowed"],masks["stop_allowed"],masks["target_allowed"]))
        logits=logits.masked_fill(~allowed,-torch.inf)
        locations=torch.cat((z,account.new_full((n,),-0.2),account.new_zeros(h),
                             account.new_full((h,),-3.5),account.new_full((h,),-3.0)))
        scales=torch.full_like(locations,0.25)
        return logits,locations,scales,z.squeeze(0)  # Zero dummy critic, not a trained value.

def sample_hybrid(logits,locations,scales,n,h):
    # All sampled scalars remain on-device; no token.item() in the clock loop.
    token=torch.multinomial(torch.softmax(logits,dim=0),1).squeeze(0)
    kind=torch.where((token>0)&(token<=n),1,
         torch.where(token>=1+n+h,2,0))
    location=locations.gather(0,token.reshape(1)).squeeze(0)
    scale=scales.gather(0,token.reshape(1)).squeeze(0)
    latent=torch.where(kind>0,location+scale*torch.randn((),device=logits.device),0.0)
    parameter=torch.where(kind==1,latent.sigmoid(),torch.where(kind==2,F.softplus(latent),0.0))
    normal=-0.5*((latent-location)/scale).square()-scale.log()-0.5*math.log(2*math.pi)
    jacobian=torch.where(kind==1,F.logsigmoid(latent)+F.logsigmoid(-latent),F.logsigmoid(latent))
    likelihood=torch.log_softmax(logits,dim=0).gather(0,token.reshape(1)).squeeze(0)+torch.where(kind>0,normal-jacobian,0.0)
    return token,latent,parameter,likelihood

def decode_sample(token,parameter,held,n):
    # Decode using THIS sample's held axis, never the axis after its fill.
    h=held.numel()
    enter=(token>0)&(token<=n)
    held_action=token>=1+n
    group=torch.div(token-1-n,max(h,1),rounding_mode="floor")
    slot=torch.remainder(token-1-n,max(h,1))
    padded=torch.cat((held,held.new_zeros(1)))  # Safe H=0 gather; masked branch cannot act.
    listing=torch.where(enter,(token-1).clamp(0,n-1),padded.gather(0,slot.reshape(1)).squeeze(0))
    code=torch.where(enter,1,torch.where(held_action,group+2,0)) # 0 hold,1 enter,2 exit,3 stop,4 target.
    return code,listing,parameter


## Tensor environment: proposals, reservations, partial fills and outcomes

ENTER sizes from free cash and current completed mark. A research share cap
matches V6's imported V2 price bands. Persistent BUY remainders reserve their
decision-price cap; fills require VWAP <= cap, avoiding overspending on a gap.
EXIT cancels unfinished acquisition. STOP/TARGET set once, relative to confirmed
average entry, using V6 positive log distance and explicit price-increment flooring.

Bracket hits use completed closes and queue next-interval exits; they cannot sell
in the first entry interval. One side per listing shares one participation budget.
No fees/slippage/LULD penalty or terminal liquidation is fabricated. This differs
from V6 quote/IOC/price-sidecar execution and must have a new environment identity.


In [ ]:
def submit_sample(state,code,listing,parameter,mark,second):
    n=mark.numel()
    selected=torch.arange(n,device=mark.device)==listing
    reserved=torch.where(state["side"]==1,state["remaining"]*state["entry_cap"],0.0).sum()
    free=(state["cash"]-reserved).clamp_min(0)
    share_cap=torch.where(mark<1,40000,torch.where(mark<=5,35000,
        torch.where(mark<=10,30000,torch.where(mark<=20,25000,torch.where(mark<=50,20000,15000)))))
    requested=torch.minimum(torch.floor(free*parameter/mark).to(torch.long),share_cap)
    enter=selected&(code==1)&(state["quantity"]==0)&(state["remaining"]==0)&(requested>0)
    exit_=selected&(code==2)&(state["quantity"]>0)&(state["side"]!=2)
    average=state["book_cost"]/state["quantity"].clamp_min(1)
    proposed=average*torch.exp(torch.where(code==3,-parameter,parameter).to(MONEY_DTYPE))
    proposed=torch.floor(proposed/PRICE_INCREMENT+1e-9)*PRICE_INCREMENT
    set_stop=selected&(code==3)&(state["quantity"]>0)&(state["stop"]==0)
    set_stop=set_stop&(proposed>0)&(proposed<average)&torch.isfinite(proposed)
    set_target=selected&(code==4)&(state["quantity"]>0)&(state["target"]==0)
    set_target=set_target&(proposed>average)&torch.isfinite(proposed)
    accepted=(enter|exit_|set_stop|set_target).any()
    updated={**state,
        "side":torch.where(enter,1,torch.where(exit_,2,state["side"])),
        "remaining":torch.where(enter,requested,torch.where(exit_,state["quantity"],state["remaining"])),
        "entry_cap":torch.where(enter,mark,torch.where(exit_,0.0,state["entry_cap"])),
        "request_fraction":torch.where(enter,parameter,torch.where(exit_,1.0,state["request_fraction"])),
        "stop":torch.where(set_stop,proposed,state["stop"]),
        "target":torch.where(set_target,proposed,state["target"]),
        "last_action_second":torch.where((set_stop|set_target).any(),second,state["last_action_second"])}
    return updated,accepted  # Real V6 must remember immediate SET_STOP/TARGET outcomes.

def execute_interval(state,vwap,volume,mark,second):
    old_qty=state["quantity"]
    capacity=torch.floor(volume*PARTICIPATION).to(torch.long)
    buy=torch.where((state["side"]==1)&(vwap<=state["entry_cap"])&(volume>0),
                    torch.minimum(state["remaining"],capacity),0)
    sell=torch.where((state["side"]==2)&(volume>0),
                     torch.minimum(torch.minimum(state["remaining"],capacity),old_qty),0)
    average=state["book_cost"]/old_qty.clamp_min(1)
    quantity=old_qty+buy-sell
    remaining=state["remaining"]-buy-sell
    cash=state["cash"]+(sell*vwap-buy*vwap).sum()
    realized_delta=(sell*(vwap-average)).sum()
    book=state["book_cost"]+buy*vwap-sell*average
    stop=torch.where(quantity>0,state["stop"],0.0)
    target=torch.where(quantity>0,state["target"],0.0)
    # End-of-interval risk triggers cause FUTURE sells, not same-bar invented fills.
    hit=(old_qty>0)&(quantity>0)&(state["side"]!=2)&(
        ((stop>0)&(mark<=stop))|((target>0)&(mark>=target)))
    side=torch.where(hit,2,torch.where(remaining>0,state["side"],0))
    remaining=torch.where(hit,quantity,remaining)
    entry_cap=torch.where(side==1,state["entry_cap"],0.0)
    entry_second=torch.where((old_qty==0)&(buy>0),second,state["entry_second"])
    entry_second=torch.where(quantity>0,entry_second,-1)
    outcome=dict(buy=buy,sell=sell,filled=buy+sell,requested_fraction=state["request_fraction"],
                 filled_fraction=(buy+sell)/state["remaining"].clamp_min(1),
                 per_listing_realized=sell*(vwap-average),realized_delta=realized_delta,capacity=capacity)
    return {**state,"quantity":quantity,"book_cost":torch.where(quantity>0,book,0.0),
        "cash":cash,"realized":state["realized"]+realized_delta,"remaining":remaining,
        "side":side,"entry_cap":entry_cap,"stop":stop,"target":target,"entry_second":entry_second,
        "request_fraction":torch.where(hit,1.0,torch.where(remaining>0,state["request_fraction"],0.0)),
        "last_action_second":torch.where((buy+sell).sum()>0,second,state["last_action_second"])},outcome


## Collection and timing

Only the clock loop is sequential. Every listing update is a tensor operation.
Inference, sample, submission and next-interval execution run once per second.
No `.item()`, CPU conversion or DataFrame operation is used in that loop. Compact
`nonzero` still has a CUDA synchronization cost, explicitly included in wall time.

CUDA is asynchronous: synchronize BEFORE starting and AFTER the full run.
Per-inference CUDA events measure device execution; CPU fallback uses perf_counter.
Output assembly is included in full wall time; generation and printing are not.
No autograd graphs are retained during collection. PPO reconstruction later runs
with gradients; environment fills are not differentiated.


In [ ]:
@torch.no_grad()
def collect(tape,initial_cash=10000.0,keep_trace=False):
    device=tape["mark"].device
    n=tape["mark"].shape[1]
    seconds=tape["mark"].shape[0]-1
    provider=DummyV6Outputs().to(device).eval()
    state=zeros_state(n,initial_cash,device)
    token_rows=[];latent_rows=[];likelihood_rows=[];value_rows=[];listing_rows=[]
    reward_rows=[];trace=[];equity_rows=[];filled_rows=[];held_axes=[]
    events=[];cpu_inference_seconds=0.0
    if device.type=="cuda": torch.cuda.synchronize(device)
    started=perf_counter()
    for t in range(seconds):
        second=tape["clocks"][t]
        mark=tape["mark"][t]
        held,account,features,masks=observe(state,mark,second)
        before=state["cash"]+(state["quantity"]*mark).sum()
        if device.type=="cuda":
            begin=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
            begin.record()
        else:
            begin=perf_counter()
        logits,locations,scales,value=provider(n,held,account,features,masks)
        if device.type=="cuda":
            end.record();events.append((begin,end))
        else:
            cpu_inference_seconds+=perf_counter()-begin
        token,latent,parameter,old_log_prob=sample_hybrid(logits,locations,scales,n,held.numel())
        code,listing,parameter=decode_sample(token,parameter,held,n)
        state,accepted=submit_sample(state,code,listing,parameter,mark,second)
        state,outcome=execute_interval(state,tape["vwap"][t+1],tape["volume"][t+1],
                                       tape["mark"][t+1],tape["clocks"][t+1])
        after=state["cash"]+(state["quantity"]*tape["mark"][t+1]).sum()
        token_rows.append(token);latent_rows.append(latent)
        likelihood_rows.append(old_log_prob);value_rows.append(value)
        listing_rows.append(torch.where(code>0,listing,-1))
        reward_rows.append((after-before)/initial_cash)
        equity_rows.append(after);filled_rows.append(outcome["filled"].sum())
        if keep_trace:
            # Debug evidence only; real PPO stores compact outcomes/data references,
            # not a full [T,N] account replay on every rollout.
            held_axes.append(held.clone())
            trace.append(dict(quantity=state["quantity"].clone(),cash=state["cash"].clone(),
                              remaining=state["remaining"].clone(),capacity=outcome["capacity"].clone(),
                              filled=outcome["filled"].clone()))
    # Preserve integer token identity and exact sampled float evidence separately.
    packets=dict(token=torch.stack(token_rows),latent=torch.stack(latent_rows),
        old_log_prob=torch.stack(likelihood_rows),old_value=torch.stack(value_rows),
        listing_index=torch.stack(listing_rows))
    rewards=torch.stack(reward_rows)
    equity=torch.stack(equity_rows)
    filled=torch.stack(filled_rows)
    if device.type=="cuda": torch.cuda.synchronize(device)
    elapsed=perf_counter()-started
    dummy_seconds=(sum(a.elapsed_time(b) for a,b in events)/1000
                   if device.type=="cuda" else cpu_inference_seconds)
    return dict(state=state,packets=packets,rewards=rewards,equity=equity,filled=filled,
        elapsed_seconds=torch.ones_like(rewards),terminated=torch.zeros_like(rewards,dtype=torch.bool),
        bootstrap=rewards.new_zeros(()),held_axes=held_axes,trace=trace,
        wall_seconds=elapsed,dummy_inference_seconds=dummy_seconds,inference_calls=seconds,
        device=str(device))

torch.manual_seed(7)
demo_tape=make_tape(8,30,DEVICE)
demo=collect(demo_tape,keep_trace=True)
print("Typed packets: token, latent, old_log_prob, dummy_value, resolved listing index (-1 for HOLD)")
print({name:values[:10].cpu() for name,values in demo["packets"].items()})
print({"device":demo["device"],"inference_calls":demo["inference_calls"],
       "wall_seconds":demo["wall_seconds"],"filled_shares":int(demo["filled"].sum().cpu()),
       "final_equity":float(demo["equity"][-1].cpu())})


## V6 compatibility and execution checks

These import only the existing pure distribution/action contracts, not training
or live execution. Likelihoods are compared with `HybridDistribution.log_prob`
for HOLD/ENTER/EXIT/STOP/TARGET. Tick rounding is checked against the existing
`decode_proposal`. No V6 file, policy checkpoint, dataset or training gate changes.


In [ ]:
from research.rl_trading.v6.actor_critic import HybridDistribution
from research.rl_trading.v6.action_adapter import decode_proposal
from research.rl_trading.v6.rl_training import PolicyRollout

# Fixed pre-decision N=4,H=2 fixture, including both held-slot mappings.
n,h=4,2
logits=torch.zeros(1+n+3*h)
locations=torch.full_like(logits,-3.0);scales=torch.full_like(logits,0.25)
dist=HybridDistribution(logits,locations,scales,n,h)
latent=torch.tensor(-3.0)
def tensor_likelihood(token,latent):
    kind=torch.where((token>0)&(token<=n),1,torch.where(token>=1+n+h,2,0))
    normal=-0.5*((latent-locations[token])/scales[token]).square()-scales[token].log()-0.5*math.log(2*math.pi)
    jac=torch.where(kind==1,F.logsigmoid(latent)+F.logsigmoid(-latent),F.logsigmoid(latent))
    return torch.log_softmax(logits,0)[token]+torch.where(kind>0,normal-jac,0.0)

# This is a bounded contract-fixture loop, never a ticker-update calculation.
for token in (0,1,1+n,1+n+h,1+n+2*h):
    assert torch.allclose(tensor_likelihood(torch.tensor(token),latent),dist.log_prob(token,latent),atol=1e-6)
held=torch.tensor([1,3],dtype=torch.long)
stop_token=1+n+h
chosen=torch.full_like(logits,-torch.inf);chosen[stop_token]=0
proposal=decode_proposal(chosen,torch.full((n,),0.5),torch.full((h,),-3.0),
    torch.full((h,),-3.0),held_index=held,entry_prices=(20.0,30.0),held_tick_sizes=(PRICE_INCREMENT,PRICE_INCREMENT))
code_,listing_,_=decode_sample(torch.tensor(stop_token),F.softplus(latent),held,n)
assert int(code_)==3 and int(listing_)==proposal.listing_index==1
expected=torch.floor(torch.tensor(20.0,dtype=MONEY_DTYPE)*torch.exp(-F.softplus(latent).to(MONEY_DTYPE))
                     /PRICE_INCREMENT+1e-9)*PRICE_INCREMENT
assert abs(float(expected)-proposal.price)<1e-9

assert demo["inference_calls"]==30
assert not demo["rewards"].requires_grad and not demo["packets"]["old_log_prob"].requires_grad
assert torch.allclose(demo["rewards"].sum(),(demo["equity"][-1]-10000.0)/10000.0,atol=1e-9)
assert torch.all(demo["state"]["quantity"]>=0) and demo["state"]["cash"]>=-1e-8
for row in demo["trace"]:
    assert torch.all(row["filled"]<=row["capacity"]) and torch.all(row["remaining"]>=0)

# Fixed seed and identical past must give identical sampled evidence before a
# changed future interval. This does not demand CPU/CUDA RNG sequence equivalence.
future_tape={key:value.clone() for key,value in demo_tape.items()}
future_tape["mark"][16:]+=1.0;future_tape["vwap"][16:]+=1.0
torch.manual_seed(7)
future=collect(future_tape,keep_trace=False)
assert torch.equal(demo["packets"]["token"][:15],future["packets"]["token"][:15])
assert torch.allclose(demo["rewards"][:15],future["rewards"][:15])

# A forced ENTER cannot create a holding before the next interval. Zero volume
# cannot fill; a later positive interval fills at most its participation capacity.
s=zeros_state(4,1000.0,DEVICE);price=torch.full((4,),20.0,device=DEVICE,dtype=MONEY_DTYPE)
s,_=submit_sample(s,torch.tensor(1,device=DEVICE),torch.tensor(2,device=DEVICE),
                  torch.tensor(0.5,device=DEVICE),price,torch.tensor(0,device=DEVICE))
assert s["quantity"].sum()==0 and s["remaining"][2]==25
s,o=execute_interval(s,price,torch.zeros_like(price),price,torch.tensor(1,device=DEVICE))
assert o["filled"].sum()==0 and s["remaining"][2]==25
s,o=execute_interval(s,price,torch.full_like(price,20.0),price,torch.tensor(2,device=DEVICE))
assert s["quantity"][2]==2 and s["remaining"][2]==23
rollout=PolicyRollout(demo["packets"]["old_log_prob"],demo["packets"]["old_value"],
    demo["rewards"],demo["terminated"],demo["elapsed_seconds"],demo["bootstrap"])
advantages,returns=rollout.targets()
assert advantages.shape==(30,) and torch.isfinite(advantages).all() and torch.isfinite(returns).all()
print("PASS: V6 likelihood/action mapping, detached reward, delayed/partial fills and capacity.")
print("PASS: typed sample evidence accepted by V6 PolicyRollout/elapsed GAE (zero dummy critic).")


## Stress test: one dummy inference per 1s, with explicit real-model budget

Data generation/initial transfer are measured separately. The measured full loop
includes observation/compaction, dummy inference, stochastic sampling, proposal
submission, execution, reward and compact result assembly. It excludes full V6
market inference, PPO backward/update, files, fees and production journaling.

`ASSUMED_V6_INFERENCE_MS` is an **illustrative input**, not a measurement. The
projection replaces measured dummy forward time with `seconds * assumed_ms`,
assuming serialized inference and replay. CUDA-event time does not capture all
host launch overhead, so this is only an approximate budget. Replace the input
with measured full V6 encode/rank/attention/decoder/critic/outcome-memory time on
the same hardware and N/R/H axes. One inference per second never means sleeping
one wall-clock second during training.

Default: 1,000 listings for 10 minutes = 600 one-second action/execution slots,
600 dummy inference calls and 601 tape boundaries including the initial state.
The previous two-minute default had 120 slots and 120 inference calls.


In [ ]:
N_LISTINGS=1_000
START_TIME="2026-10-01T08:00:00+00:00"
END_TIME="2026-10-01T08:10:00+00:00"
ASSUMED_V6_INFERENCE_MS=5.0  # Illustrative, replace with measured full V6 forward cost.
MAX_ESTIMATED_GIB=1.0
start=datetime.fromisoformat(START_TIME);end=datetime.fromisoformat(END_TIME)
assert start.tzinfo is not None and end.tzinfo is not None
delta=end.astimezone(timezone.utc)-start.astimezone(timezone.utc)
us=(delta.days*86400+delta.seconds)*1_000_000+delta.microseconds
assert us>0 and us%1_000_000==0
SECONDS=us//1_000_000
assert type(N_LISTINGS) is int and N_LISTINGS>0
assert math.isfinite(ASSUMED_V6_INFERENCE_MS) and ASSUMED_V6_INFERENCE_MS>=0
estimated_gib=(SECONDS+1)*N_LISTINGS*8*3*3/(1024**3)
if estimated_gib>MAX_ESTIMATED_GIB:
    raise MemoryError(f"Estimated resident tape plus workspace {estimated_gib:.2f} GiB exceeds guard")
if DEVICE.type=="cuda": torch.cuda.synchronize(DEVICE)
generation_start=perf_counter()
pressure_tape=make_tape(N_LISTINGS,SECONDS,DEVICE)
if DEVICE.type=="cuda": torch.cuda.synchronize(DEVICE)
generation_seconds=perf_counter()-generation_start
torch.manual_seed(17)
pressure=collect(pressure_tape,keep_trace=False)
projected=max(0.0,pressure["wall_seconds"]-pressure["dummy_inference_seconds"])+SECONDS*ASSUMED_V6_INFERENCE_MS/1000
print({"device":pressure["device"],"torch":torch.__version__,"listings":N_LISTINGS,
       "simulated_seconds":SECONDS,"time_slots":SECONDS,"tape_boundaries":SECONDS+1,"resolution_seconds":1,"inference_calls":pressure["inference_calls"],
       "generation_and_transfer_seconds":generation_seconds,
       "measured_replay_with_dummy_forward_seconds":pressure["wall_seconds"],
       "measured_dummy_forward_seconds":pressure["dummy_inference_seconds"],
       "measured_simulated_seconds_per_wall_second":SECONDS/pressure["wall_seconds"],
       "assumed_v6_forward_ms":ASSUMED_V6_INFERENCE_MS,
       "projected_real_forward_plus_replay_seconds_NOT_MEASURED":projected,
       "filled_shares":int(pressure["filled"].sum().cpu())})
assert pressure["inference_calls"]==SECONDS and pressure["packets"]["token"].shape==(SECONDS,)
assert pressure["packets"]["token"].dtype==torch.long
assert torch.isfinite(pressure["rewards"]).all() and torch.isfinite(pressure["packets"]["old_log_prob"]).all()


### Measured local example: 10-minute session

Measured on 2026-09-30 with Torch 2.12.0+cpu, one CPU thread and no CUDA:
1,000 listings, 600 one-second slots, 601 tape boundaries and 600 dummy inference
calls. Initial full resident replay took **0.711 s**; three additional runs took
**0.924, 0.886 and 0.926 s** (repeat-run median **0.924 s**). Dummy tape generation
took **0.008 s**, separately. These are local examples, not GPU speed claims or
real V6 benchmarks. Execution correctness assertions passed in this run.

The earlier two-minute default had 120 slots and 120 calls; its roughly 0.13 s
result likewise used CPU and dummy outputs. Ten minutes refers to simulated
market time: the collector does not sleep between one-second clocks.

With an illustrative **5 ms full V6 forward per call**, replacing dummy-forward
time gives approximately **3.7-3.9 s** for 600 calls plus replay on these runs.
That is an unmeasured serialized budget, not a promise: actual V6 inference,
recurrent state handling and observation construction must be benchmarked on the
target GPU. This timing excludes real ARTE loading/preparation, PPO backward and
optimizer updates, and production execution features absent from this prototype.


## Real execution data: certified ARTE sources and the GPU adapter

This notebook still generates dummy data. The following is the integration contract
for a future loader, not an implemented database connection or verified coverage.
Use ClickHouse database `arte`, with build/day/ticker/stage-attempt selections
pinned through the existing certification ledger. Rows existing in a table alone
do not establish certified coverage. Verify `live_market_ssd` placement using the
existing readers; do not fall back to raw SIP files or unpinned attempts.

| Source | Purpose in the approximate simulator |
| --- | --- |
| `arte.liquidity_100ms_v1` | Eligible execution volume/notional, quotes and quote validity. Aggregate certified 100 ms rows into completed 1 s execution intervals. |
| `arte.bars_v1` | Valid OHLC/close prices for marking and optional extrema-based bracket checks, with the chosen `resolution_ms`. |
| `arte.liquidity_execution_price_100ms_v1` | Optional execution volume by price for more selective passive-limit capacity. Requires `arte.liquidity_execution_price_coverage_v1` certification; traded volume does not prove our order would fill. |

Verified code paths (repository-relative):
- `research/rl_trading/v6/environment_source.py::ArteExecutionSource` reads pinned
  100 ms liquidity quotes and 100 ms bars, and certifies optional target-price evidence.
- `research/rl_trading/v1/bracket_source.py::broker_attempts` pins broker attempts;
  `assert_liquidity_storage` checks table/part placement.
- `research/rl_trading/v6/bracket_evidence.py` and
  `src/backend/backtest_liquidity_price.py` own certified price-level evidence.
- `pipelines/market_sip/events/market_day_sql.py` defines the bar/liquidity schemas.

### Projection into this notebook's tensor contract

For each fixed listing identity and execution interval, calculate:

    volume[t, n] = sum(execution_volume in this interval)
    vwap[t, n]   = sum(execution_notional) / volume[t, n], if volume > 0
    mark[t, n]   = valid completed bar close_int / 10_000

Use interval sums, not the liquidity table's cumulative-session `execution_vwap`.
The participation fraction times eligible interval volume bounds partial fills;
VWAP is an approximation, not a quote/queue-level broker execution guarantee.
A valid zero-volume interval has zero capacity. Missing/invalid evidence is not a
zero-volume interval: preserve validity/coverage masks and fail closed or explicitly
defer affected fills. The dummy executor currently lacks these masks, so a real
adapter must extend it before supplying incomplete historical data. Never fill
at a fabricated zero VWAP. Mark carry-forward needs an explicit freshness policy.

ARTE `bucket_index` is relative to **America/New_York local midnight** for
`session_date`, not to session start. Reconstruct UTC boundary microseconds using
that day's timezone-aware midnight and `(bucket_index + 1) * resolution_ms * 1000`.
Aggregate ten aligned 100 ms buckets per 1 s interval, and check certified coverage,
duplicates, ordering, source validity and stable listing-to-tensor mapping. Do not
rely on today's ticker identity to resolve a historical listing.

`mark`, `vwap`, `volume` are device-resident `[T+1, N]` tensors here. Index 0 is
the initial observation boundary; indices 1..T describe successive execution
intervals. A proposal sampled at boundary t first consumes interval t+1, and its
confirmed result becomes visible at boundary t+1. Historical future intervals may
be resident in GPU memory but must never enter earlier policy observations.
Load/certify/aggregate once outside timed resident replay; measure that preparation
separately if reporting cold end-to-end throughput.

Keep quotes/freshness, OHLC extrema, validity masks, price-level capacity, source
provenance and identity metadata alongside the three minimal tensors when needed.
The executable demonstration currently uses close-based brackets; OHLC triggers
and price-level fills are future extensions with their own documented semantics.
Existing V6 uses quote-bound IOC execution. Replacing it with persistent 1 s
participation/VWAP fills requires an explicit environment version and acceptance.


## Integration handoff: what an agent must preserve or change explicitly

1. Keep this as a new approximate environment until independently accepted.
   Pin an environment/fill-model version. Existing V6 quote/IOC, costs, LULD,
   share caps, terminal penalties and replay ledgers are not silently equivalent.
2. Replace only `DummyV6Outputs.forward` with the causal V6 model call. Inputs must
   come from completed bank rows and reconstructed action memory. Encoder windows,
   sparse actual-candle updates and ranked attention are deliberately absent here.
3. Build the seven account and eleven held fields/masks on device. Validate against
   `observe_account`, including reservation, price freshness and halt semantics.
   Preserve compact H token axes; padded H needs an explicit invertible mapping.
4. Collection must use policy samples, not argmax or teacher actions. Retain token,
   latent, old log probability/value, held mapping, causal masks, data-row references
   and execution-outcome records. Dummy packets here are numeric evidence, not a
   complete recurrent PPO reconstruction buffer. Do not replace raw samples with
   rounded orders or partial-fill fractions in the PPO likelihood.
5. Wire actual outcomes to `remember_execution` before next inference; update
   immediate bracket outcomes at submission. Quantities/cash change only on fills.
   Do not differentiate through the environment or reuse future actor observations.
6. Preserve `collect_session`'s reward scaling and restore costs/penalties exactly
   once. Our reward telescopes marked equity/initial cash with no extra costs.
   At truncation, obtain actual next-state bootstrap; our zero is valid ONLY for
   the dummy constant-zero critic. At a genuine session end, implement audited
   forced-exit scheduling/remaining-exposure behavior, then mark terminal.
7. Reconstruct policy recurrent states chronologically for PPO, with causal burn-in
   and bounded BPTT. Verify first reconstructed joint likelihood equals collection
   before updating. Existing `PolicyRollout`/`elapsed_gae`/`update_on_policy` own the
   target/loss contract; this notebook does not implement an optimizer step.
8. Decide one proposal per second versus existing up-to-64 cadence before merging.
   Count actual inferences, including bootstrap-only evaluations, in throughput.
   If retaining multiple same-clock samples, preserve zero elapsed-time semantics.
9. Benchmark CPU, eager CUDA and later compiled CUDA with matched trajectories,
   precision and outputs. `torch.compile`/CUDA graphs are future optimizations,
   not enabled or measured here. Compact nonzero, dynamic H and host assertions in
   existing V6 interfaces can force synchronization; replacing those contracts is
   separate from moving arithmetic to CUDA. More independent environments can
   improve utilization, but require separate cash/liquidity and ragged H handling.

Resident training data avoids repeated I/O/transfers, but PPO trajectories remain
on-policy: do not cache actions/outcomes across changed weights. Frozen market-only
embeddings can be cached; trainable encoder states must be reconstructed with the
current weights for update. All old likelihoods and returns stay detached.

Outputs are shown in memory. Save generated reports/checkpoints only under
`D:/TradingML/runtimes/vectorized_backtest/`. No actual V6 training was launched.
